# Explorador de Tablas

Ingresa el nombre de cualquier tabla de la base de datos y visualiza sus datos en formato tabla.

In [28]:
from pathlib import Path
import subprocess
import tempfile

import pandas as pd
from IPython.display import display, HTML

DB_PATH = "localhost:/Users/wilsonjonatan/Documents/bases 1 2026/violencia guate/sql/db/violencia_guate.fdb"
DB_USER = "sysdba"
DB_PASSWORD = "masterkey"
ISQL_PATH = "/Library/Frameworks/Firebird.framework/Resources/bin/isql"

In [29]:
def run_query(sql: str) -> pd.DataFrame:
    query = sql.strip().rstrip(";") + ";"
    script = f"SET HEADING OFF;\nSET LIST ON;\n{query}\n"

    with tempfile.NamedTemporaryFile("w", suffix=".sql", delete=False, encoding="utf-8") as tmp:
        tmp.write(script)
        tmp_path = tmp.name

    try:
        result = subprocess.run(
            [ISQL_PATH, "-user", DB_USER, "-password", DB_PASSWORD, DB_PATH, "-q", "-i", tmp_path],
            capture_output=True,
            text=True,
        )
    finally:
        Path(tmp_path).unlink(missing_ok=True)

    output = "\n".join(part for part in (result.stdout, result.stderr) if part)
    rows = []
    current = {}

    for raw_line in output.splitlines():
        line = raw_line.strip()
        if not line:
            if current:
                rows.append(current)
                current = {}
            continue
        if line.startswith("SQL>") or line.startswith("DatabaseError") or line.startswith("Warning"):
            continue

        parts = line.split(None, 1)
        if len(parts) == 2:
            key, value = parts
            current[key.lower()] = value.strip()

    if current:
        rows.append(current)

    df = pd.DataFrame(rows)
    for column in df.columns:
        numeric = pd.to_numeric(df[column], errors="coerce")
        if len(df[column]) > 0 and numeric.notna().all():
            df[column] = numeric
    return df


def get_table_info(tabla: str) -> tuple:
    """Obtiene el conteo y primeras filas de una tabla."""
    sql_count = f"SELECT COUNT(*) AS total FROM {tabla};"
    sql_select = f"SELECT * FROM {tabla};"
    
    try:
        df_count = run_query(sql_count)
        if df_count.empty:
            return None, 0
        
        total = int(df_count.iloc[0, 0])
        df_data = run_query(sql_select)
        return df_data, total
    except Exception as e:
        print(f"Error: {e}")
        return None, 0

In [ ]:
# ========== CONFIGURACIÓN: escribe aquí cualquier consulta SQL ==========
CONSULTA_SQL = """


SELECT
    EXTRACT(YEAR FROM pd.fecha_diagnostico) AS anio,
    d.nombre AS departamento,
    CASE
        WHEN di.nombre CONTAINING 'Dengue' THEN 'DENGUE GRAVE'
        ELSE 'DENGUE'
    END AS tipo_dengue,
    SUM(COALESCE(pd.cantidad, 0)) AS total_casos
FROM persona_diagnostico pd
JOIN diagnostico di ON di.id_diagnostico = pd.id_diagnostico
JOIN municipio m ON m.id_municipio = pd.id_municipio
JOIN departamento d ON d.id_departamento = m.id_departamento
WHERE EXTRACT(YEAR FROM pd.fecha_diagnostico) BETWEEN 2012 AND 2024
  AND di.nombre CONTAINING 'DENGUE'
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3;


"""
# =======================================================================

print("\n📊 Ejecutando consulta:")
print(CONSULTA_SQL)
print("=" * 60)

try:
    df = run_query(CONSULTA_SQL)

    if df.empty:
        print("⚠️ La consulta no devolvió filas.")
    else:
        print(f"✓ Filas obtenidas: {len(df)}")
        print(f"✓ Columnas: {len(df.columns)}")
        print(f"\nEncabezados: {', '.join(df.columns)}")
        print("-" * 60)
        display(df.head(MAX_FILAS_MOSTRAR))

except Exception as e:
    print(f"❌ Error al ejecutar la consulta: {e}")


📊 Ejecutando consulta:



SELECT
    EXTRACT(YEAR FROM pd.fecha_diagnostico) AS anio,
    d.nombre AS departamento,
    CASE
        WHEN di.nombre CONTAINING 'Dengue Grave' THEN 'DENGUE GRAVE'
        ELSE 'DENGUE'
    END AS tipo_dengue,
    SUM(COALESCE(pd.cantidad, 0)) AS total_casos
FROM persona_diagnostico pd
JOIN diagnostico di ON di.id_diagnostico = pd.id_diagnostico
JOIN municipio m ON m.id_municipio = pd.id_municipio
JOIN departamento d ON d.id_departamento = m.id_departamento
WHERE EXTRACT(YEAR FROM pd.fecha_diagnostico) BETWEEN 2012 AND 2024
  AND di.nombre CONTAINING 'DENGUE'
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3;



✓ Filas obtenidas: 289
✓ Columnas: 4

Encabezados: anio, departamento, tipo_dengue, total_casos
------------------------------------------------------------


,anio,departamento,tipo_dengue,total_casos
0,2012,Alta Verapaz,DENGUE,751
1,2012,Baja Verapaz,DENGUE,176
2,2012,Chimaltenango,DENGUE,23
3,2012,Chiquimula,DENGUE,835
4,2012,El Progreso,DENGUE,233
...,...,...,...,...
284,2024,Santa Rosa,DENGUE,5604
285,2024,Sololá,DENGUE,5183
286,2024,Suchitepéquez,DENGUE,5690
287,2024,Totonicapán,DENGUE,39
